In [ ]:
import yfinance as yf
from pandas import Timestamp
from tqdm.notebook import tqdm
import pandas as pd
import os
import numpy as np
import seaborn as sns


In [ ]:
seg = "Monthly"
level = "COLLLevel-MEAN"
k = 100
min_sim = 0.0
path = f"../graphsV2/{level}/"

last_graph_path = f"{path}/(2021, 3)/edge_list.csv"
last_graph_info = f"{path}/(2021, 3)/metadata.csv"
SOGLIA=.5

# Calcolo degree semplice e pesato per collection

In [ ]:
last_graph=pd.read_csv(last_graph_path, index_col=0)
last_graph = last_graph[last_graph.weight >= SOGLIA]
last_info=pd.read_csv(last_graph_info,index_col=0).fillna(0)
last_info.index=last_info.id.values

out_edges={}   #nft:[edges]
in_edges={}   #nft:[edges]


for node in last_info.id.values:
    if node not in out_edges:
        out_edges[node]=pd.DataFrame()
    if node not in in_edges:
        in_edges[node]=pd.DataFrame()


for node_source,edges in last_graph.groupby("source"):
    out_edges[node_source]=edges

for node_target,edges in last_graph.groupby("target"):
    in_edges[node_target]=edges
    

out_degrees={k:len(v) for k,v in out_edges.items()}
in_degrees={k:len(v) for k,v in in_edges.items()}    

out_degrees_w={k:0 if len(v)==0 else sum([e["weight"] for _,e in v.iterrows()]) for k,v in out_edges.items()}
in_degrees_w={k:0 if len(v)==0 else sum([e["weight"] for _,e in v.iterrows()]) for k,v in in_edges.items()}


# Calcolo del copy-rate
assert round(sum(out_degrees_w.values()), 3) == round(sum(in_degrees_w.values()), 3)

copy_rate = {}
insp_rate = {}
overall_out_deg_w = 0
overall_in_deg_w = 0
for node in [n for n in last_info.id.values if (out_degrees[n] + in_degrees[n]) > 0]: # li consideriamo solo se non sono isolati
    c_r = 0
    i_r = 0
    overall_out_deg_w += out_degrees_w[node]
    overall_in_deg_w += in_degrees_w[node]

    if in_degrees_w[node]!=0:
        c_r=out_degrees_w[node]/in_degrees_w[node]
    else:
        c_r = out_degrees_w[node]
    copy_rate[node]=c_r

    if out_degrees_w[node]!=0:
        i_r = in_degrees_w[node]/out_degrees_w[node]
    else:
        i_r = in_degrees_w[node]
    insp_rate[node] = i_r


In [ ]:
# Checking the most relevant NFTs (collections) in terms of copy-rate
# sorted_copy_rate = {k:v for k, v in sorted(copy_rate.items(), key=lambda x:x[1], reverse=True)}
copy_rate_df = pd.DataFrame(copy_rate.items(), columns=['collection', 'c_rate'])

# Normalize by the corresponding out-degree (scale by out_deg/overall_out_deg)
copy_rate_df['scale_copy'] = copy_rate_df['collection'].apply(lambda x: out_degrees_w[x] / overall_out_deg_w)
copy_rate_df['c_rate'] = copy_rate_df['c_rate'] * copy_rate_df['scale_copy']

# Integrating inspiration rates
copy_rate_df['scale_insp'] = copy_rate_df['collection'].apply(lambda x: in_degrees_w[x] / overall_in_deg_w)
copy_rate_df['i_rate'] = (copy_rate_df['collection'].apply(lambda x: insp_rate[x])) * copy_rate_df['scale_insp']

# Potrebbe aver senso tenere conto dei volumi, così da poter confrontare "original" e "inspired"

In [ ]:
# Getting the most "copying" collections
copy_rate_df.sort_values(by='c_rate', ascending=False).reset_index(drop=True)[['collection', 'c_rate']].head(10)

In [ ]:
# Getting the most "inspiring" collections
copy_rate_df.sort_values(by='i_rate', ascending=False).reset_index(drop=True)[['collection', 'i_rate']].head(10)

In [ ]:
last_info

In [ ]:
# Checking correlation between inspiration and volume
insp_df = copy_rate_df.sort_values(by='i_rate', ascending=False).reset_index(drop=True)[['collection', 'i_rate']]
insp_df['#tx'] = insp_df['collection'].apply(lambda x: last_info.loc[x, '#tx'])
insp_df['vol'] = insp_df['collection'].apply(lambda x: last_info.loc[x, 'volume'])
insp_df['avg'] = insp_df['collection'].apply(lambda x: last_info.loc[x, 'avg_price'])
insp_df['max'] = insp_df['collection'].apply(lambda x: last_info.loc[x, 'max_price'])
insp_df.head(10)

Vedere inspiration di "ieri" con valori di volume di "oggi"

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,5))
sns.heatmap(insp_df.corr(), vmin=-1, vmax=1, annot=True, cmap="YlGnBu")

In [ ]:
# Trying to define the "inspiration-rate" as the complementary measure of "copy-rate" (doesn't work well!)
copy_rate_df['insp_rate'] = 1-copy_rate_df['c_rate']
sorted_insp_rate_df = copy_rate_df.sort_values(by='insp_rate', ascending=False).reset_index(drop=True)
sorted_insp_rate_df.head(10)

# Calcolo degree semplice e pesato per categoria

In [ ]:
last_graph_cat=pd.read_csv(last_graph_path,index_col=0)
last_graph_cat = last_graph_cat[last_graph_cat.weight > SOGLIA]
last_info=pd.read_csv(last_graph_info,index_col=0).fillna(0)
last_info.index=last_info.id.values



#sostituisco le collection con le categorie
last_graph_cat.source = last_graph_cat.source.apply(lambda x: last_info.loc[x].category)
last_graph_cat.target = last_graph_cat.target.apply(lambda x: last_info.loc[x].category)

out_edges_cat={}   #nft:[edges]
in_edges_cat={}   #nft:[edges]


for node in last_info.category.values:
    if node not in out_edges_cat:
        out_edges_cat[node]=pd.DataFrame()
    if node not in in_edges_cat:
        in_edges_cat[node]=pd.DataFrame()

for node_source,edges in last_graph_cat.groupby("source"):
    out_edges_cat[node_source]=edges

for node_target,edges in last_graph_cat.groupby("target"):
    in_edges_cat[node_target]=edges
    


out_degrees_cat={k:len(v) for k,v in out_edges_cat.items()}
in_degrees_cat={k:len(v) for k,v in in_edges_cat.items()}    

out_degrees_w_cat={k:0 if len(v)==0 else sum([e["weight"] for _,e in v.iterrows()]) for k,v in out_edges_cat.items()}
in_degrees_w_cat={k:0 if len(v)==0 else sum([e["weight"] for _,e in v.iterrows()]) for k,v in in_edges_cat.items()}


# Calcolo del copy-rate
assert round(sum(out_degrees_w_cat.values()), 3) == round(sum(in_degrees_w_cat.values()), 3)

copy_rate_cat = {}
insp_rate_cat = {}
overall_out_deg_w_cat = 0
overall_in_deg_w_cat = 0
for node in [n for n in last_info.category.values if (out_degrees_cat[n] + in_degrees_cat[n]) > 0]: # li consideriamo solo se non sono isolati
    c_r = 0
    i_r = 0
    overall_out_deg_w_cat += out_degrees_w_cat[node]
    overall_in_deg_w_cat += in_degrees_w_cat[node]

    if in_degrees_w_cat[node]!=0:
        c_r=out_degrees_w_cat[node]/in_degrees_w_cat[node]
    else:
        c_r = out_degrees_w_cat[node]
    copy_rate_cat[node]=c_r

    if out_degrees_w_cat[node]!=0:
        i_r = in_degrees_w_cat[node]/out_degrees_w_cat[node]
    else:
        i_r = in_degrees_w_cat[node]
    insp_rate_cat[node] = i_r


In [ ]:
# Checking the most relevant NFTs (collections) in terms of copy-rate
# sorted_copy_rate = {k:v for k, v in sorted(copy_rate.items(), key=lambda x:x[1], reverse=True)}
copy_rate_df_cat = pd.DataFrame(copy_rate_cat.items(), columns=['category', 'c_rate'])

# Normalize by the corresponding out-degree (scale by out_deg/overall_out_deg)
copy_rate_df_cat['scale_copy'] = copy_rate_df_cat['category'].apply(lambda x: out_degrees_w_cat[x] / overall_out_deg_w_cat)
copy_rate_df_cat['c_rate'] = copy_rate_df_cat['c_rate'] * copy_rate_df_cat['scale_copy']

# Integrating inspiration rates
copy_rate_df_cat['scale_insp'] = copy_rate_df_cat['category'].apply(lambda x: in_degrees_w_cat[x] / overall_in_deg_w_cat)
copy_rate_df_cat['i_rate'] = (copy_rate_df_cat['category'].apply(lambda x: insp_rate_cat[x])) * copy_rate_df_cat['scale_insp']

# Potrebbe aver senso tenere conto dei volumi, così da poter confrontare "original" e "inspired"

In [ ]:
# Getting the most "copying" categories
copy_df_cat = copy_rate_df_cat.sort_values(by='c_rate', ascending=False).reset_index(drop=True)[['category', 'c_rate']]
copy_df_cat.head(10)

In [ ]:
copy_df_cat['c_rate'] = copy_df_cat['c_rate'] / copy_df_cat['c_rate'].sum()
copy_df_cat.head(10)

In [ ]:
# Getting the most "inspiring" categories
inspiring_df_cat = copy_rate_df_cat.sort_values(by='i_rate', ascending=False).reset_index(drop=True)[['category', 'i_rate']]
inspiring_df_cat.head(10)

In [ ]:
inspiring_df_cat['i_rate'] = inspiring_df_cat['i_rate'] / inspiring_df_cat['i_rate'].sum()
inspiring_df_cat.head()

### Chi arriva dopo, raggiunge gli stessi prezzi?


Metodo per la risoluzione della domanda: 

Per ogni arco copiatore -> copiato, faccio la media dei rapporti fra mean(copiato)/mean(copiatore)

se maggiore di 0 il copiato raggiunge in media prezzi più alti
se minore di 0 il copiatore raggiunge in media prezzi più alti

tale ragionamento è applicato anche a volumi, e altre statistiche.

copy-rate = grado pesato uscente / grado pesato entrante (da normalizzare)

In [ ]:
stats={
    "volume":[],
    "#tx":[],
    "avg_price":[],
    "max":[],
    "min":[],
    "std":[]
}
# for copiatore, copiato, weight,_ in last_graph.itertuples(index=False):
for _,edge in last_graph.iterrows():
    copiatore,copiato,weight=edge["source"],edge["target"],edge["weight"]
    stats_copiatore=last_info.loc[copiatore]
    stats_copiato=last_info.loc[copiato]
    
    stats["volume"].append(     (stats_copiato["volume"],stats_copiatore["volume"]))
    stats["#tx"].append(        (stats_copiato["#tx"],   stats_copiatore["#tx"]))
    stats["avg_price"].append(  (stats_copiato["avg_price"],   stats_copiatore["avg_price"]))
    stats["max"].append(        (stats_copiato["max_price"],   stats_copiatore["max_price"]))
    stats["min"].append(        (stats_copiato["min_price"],   stats_copiatore["min_price"]))
    stats["std"].append(        (stats_copiato["std_price"],   stats_copiatore["std_price"]))

for i,(k,v) in enumerate(stats.items()):
    print(f"{i}) Rapporto fra mean({k})_copiato/mean({k})_copiatore: {np.mean([x[0] for x in v]):.2f}, {np.mean([x[1] for x in v]):.2f},{np.mean([x[0] for x in v])/np.mean([x[1] for x in v]):.3f}")

# Considerazioni Lucio
- Chi arriva prima fa più volume e genera molte più transazioni
- Chi arriva prima raggiunge prezzi massimi di vendita più alti (3x) di chi "copia"
- Chi arriva dopo, si trova già con caratteristiche ben "note" e tiene un prezzo medio più alto (ma non genera gli stessi picchi e volumi)
- Chi arriva dopo ha dei minimi più evidenti (che siano feature già note?)
- Chi arriva prima ha un prezzo più ballerino, dovuto proprio alla necessità di "aprire" il mercato

# Considerazioni Davide

- Chi arriva dopo ha dei minimi più alti (4) e dei massimi più bassi (3), nonchè una deviazione standard media più bassa. Questo potrebbe indicare ciò che accade logicamente in un mercato, ovvero che con il tempo gli acquirenti diventano più abili e sono capaci quindi di rendere più stabili gli asset che acquistano/vendono attribuendo il giusto valore a mercato.

- Sembra che chi arrivi dopo riesca a trarre vantaggio dalla conoscenza di mercato, stabilendo un prezzo medio di vendita più alto (2) a fronte di volumi e #tx più modesti: indice forse che le opportunità scarseggiano e che il mercato "copiato" è "saturo"?

### Qual è il tempo necessario per raggiungere il massimo per gli originali e le copie?

In [ ]:
# Raw data containing info on prices and datetimes
data_raw = pd.read_csv("../input/Data_API.csv", usecols=['Collection', 'Unique_id_collection', 'Price_USD', 'Datetime_updated_seconds'])
data_raw['Datetime_updated_seconds'] = pd.to_datetime(data_raw['Datetime_updated_seconds'], format='%Y-%m-%d %H:%M:%S')
data_raw.head()

In [ ]:
# Computing deltas for each collection
delta_collection_up = {}
delta_collection_down = {}

for index, df_ in data_raw.groupby(by=['Collection']):
    sorted_df_ = df_.sort_values(by='Datetime_updated_seconds', ascending=True)

    first_sold = sorted_df_.iloc[0]['Datetime_updated_seconds']

    max_index = sorted_df_['Price_USD'].argmax()
    max_sold = sorted_df_.iloc[max_index]['Datetime_updated_seconds']

    min_index = sorted_df_.iloc[max_index:]['Price_USD'].argmin()
    min_sold = sorted_df_.iloc[max_index:].iloc[min_index]['Datetime_updated_seconds'] # CONTROLLARE DOPPIO INDEXING  

    assert first_sold <= max_sold, f"Assertion max: {first_sold, max_sold}"
    assert max_sold <= min_sold, f"Assertion min: {max_sold, min_sold}"

    delta_up = (max_sold-first_sold).days
    if index not in delta_collection_up:
        delta_collection_up[index] = delta_up
    else:
        raise RuntimeError("Duplicate collection found!")

    delta_down = (min_sold - max_sold).days
    if index not in delta_collection_down:
        delta_collection_down[index] = delta_down
    else:
        raise RuntimeError("Duplicate collection found!")

assert len(delta_collection_up) == len(data_raw.Collection.unique())
assert len(delta_collection_down) == len(data_raw.Collection.unique())

In [ ]:
# Checking deltas for copiatori and copiati
delta_up_copiatori = []
delta_up_copiati = []

delta_down_copiatori = []
delta_down_copiati = []

for copiatore, copiato, _,_ in last_graph.itertuples(index=False):
    if copiatore in delta_collection_up:
        delta_up_copiatori.append(delta_collection_up[copiatore])
    if copiatore in delta_collection_down:
        delta_down_copiatori.append(delta_collection_down[copiatore])

    if copiato in delta_collection_up:
        delta_up_copiati.append(delta_collection_up[copiato])
    if copiato in delta_collection_down:
        delta_down_copiati.append(delta_collection_down[copiato])

delta_up_copiatori = [x for x in delta_up_copiatori if x > 0]
delta_up_copiati = [x for x in delta_up_copiati if x > 0]

delta_down_copiatori = [x for x in delta_down_copiatori if x > 0]
delta_down_copiati = [x for x in delta_down_copiati if x > 0]

print(f"1) Avg delta-up days for copiatori: {np.mean(delta_up_copiatori)} std({np.std(delta_up_copiatori)})")
print(f"2) Avg delta-up days for copiati: {np.mean(delta_up_copiati)} std({np.std(delta_up_copiati)})")
print()
print(f"3) Avg delta-down days for copiatori: {np.mean(delta_down_copiatori)} std({np.std(delta_down_copiatori)})")
print(f"4) Avg delta-down days for copiati: {np.mean(delta_down_copiati)} std({np.std(delta_down_copiati)})")

# Considerazioni Davide

E' possibile innanzitutto osservare che i copiati hanno una stabilità del prezzo massimo nettamente superiore 3x rispetto ai copiatori 1-2. Mentre è altrettanto chiaro che i copiati hanno minimi che scendono più lentamente di circa il 30%. 

Inoltre, osservando la deviazione standard sembra indicare che il mercato dei copiati sia più ballerino, sia per quanto riguarda il tempo al next max che al next mean


### Chi arriva dopo, tiene gli stessi prezzi più a lungo o cala prima?

- Utilizzo di stdev per vedere volatilità del token
- Max delta temporale tra max-price e min-price (successivo)

In [ ]:
# Getting all NFTs for each collection
# Check se abbiamo tutti i priceUSD
coll2NFTs = {}



In [ ]:
data_sorted = data_raw.sort_values(by=['Collection', 'Datetime_updated'])
data_sorted.head()

In [ ]:
for collection in list(data_sorted.Collection.unique())[:10]:
    print("Collection: ", collection)
    data_collection = data_sorted[data_sorted.Collection == collection]
    print(data_collection)

In [ ]:
df

In [ ]:
# Method for getting the maximum and the subsequent minimum sale (with delta-time) of an NFT 

### A quante collection sei più simile?

- Degree del nodo (si applica anche a categorie mediante aggregazione)

In [ ]:
# print("Copiatori per collection")
# for k,v in copiatori_degree.items():
#     print(f"\t{k} {v}")

# print("Copiati per collection")
# for k,v in copiati_degree.items():
#     print(f"\t{k} {v}")




print("Copiatori per categorie")
for k,v in copiatori_degree_cat.items():
    print(f"\t{k} {v}")

print("Copiati per categorie")
for k,v in copiati_degree_cat.items():
    print(f"\t{k} {v}")


### Categoria soggetta a ricevere più similarità nel tempo

- Per ogni snapshot e per ogni categoria, quanti archi entranti si aggiungono ad ogni categoria, alla fine calcolo per ogni categoria la media degli archi aggiunti in ogni snapshot (e vedo se ci sono snapshot anomali per una categoria)

In [ ]:
records=[]
for snap in os.listdir(path):
    try:
        edges=pd.read_csv(f"{path}/{snap}/edge_list.csv")
        nodes=pd.read_csv(f"{path}/{snap}/curr_snap_info.csv",index_col="id")
        
    except:
        pass


### NFT (collection) silenti ma copiati, emergono (venduti) in un bull?

- NFT simili a qualcosa di già esistente: esistono NFT che sono stati acquistati per "poco" e che sono stati ri-venduti per molto durante un aumento di volume?
- Check collection date per OpenSea

### Ci sono collection inizialmente dissimili tra loro che poi diventano simili o viceversa?

- Andamento del peso sugli archi tra coppie di collection nei vari snapshot